In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:

import os
import json
import torch
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from transformers import AutoProcessor, AutoModelForCausalLM

# Paths
image_dir = "/content/drive/MyDrive/mini_project/subset/Flicker8k_1kSubset"
ground_truth_path = "/content/drive/MyDrive/mini_project/subset/subset_1k_data.json"
output_base_path = "/content/drive/MyDrive/adversarial_caption_output_batch"  # Base name for output

# Load GIT model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/git-base"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Caption generator
def generate_caption(image_tensor):
    image_pil = transforms.ToPILImage()(image_tensor.squeeze(0).cpu())
    inputs = processor(images=image_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# FGSM Attack
def fgsm_attack(image, epsilon=0.02):
    image = image.clone().detach().requires_grad_(True)
    output = image.sum()
    output.backward()
    perturbation = epsilon * image.grad.sign()
    adv_image = torch.clamp(image + perturbation, 0, 1)
    return adv_image.detach()

# PGD Attack
def pgd_attack(image, epsilon=0.2, alpha=0.01, num_iters=10):
    ori_image = image.clone().detach()
    perturbed = image.clone().detach()
    for _ in range(num_iters):
        perturbed.requires_grad = True
        loss = perturbed.sum()
        loss.backward()
        perturbed = perturbed + alpha * perturbed.grad.sign()
        perturbation = torch.clamp(perturbed - ori_image, -epsilon, epsilon)
        perturbed = torch.clamp(ori_image + perturbation, 0, 1).detach()
    return perturbed

# CW Attack (Simplified simulation)
def cw_attack(image, epsilon=0.03):
    perturbation = (torch.rand_like(image) - 0.5) * 2 * epsilon
    adv_image = torch.clamp(image + perturbation, 0, 1)
    return adv_image

# JSMA-style Attack (Saliency-based)
def jsma_attack(image, epsilon=0.05):
    image.requires_grad = True
    loss = image.sum()
    loss.backward()
    saliency = image.grad.abs()
    saliency = saliency / saliency.max()
    perturbation = epsilon * saliency
    adv_image = torch.clamp(image + perturbation, 0, 1)
    return adv_image.detach()

# One Pixel Attack (random pixel change)
def one_pixel_attack(image, num_pixels=1):
    adv_image = image.clone().detach()
    _, c, h, w = adv_image.shape
    for _ in range(num_pixels):
        x = random.randint(0, w - 1)
        y = random.randint(0, h - 1)
        for ch in range(c):
            adv_image[0, ch, y, x] = torch.rand(1).item()
    return adv_image

# Load ground truth
with open(ground_truth_path) as f:
    ground_truth_data = json.load(f)

gt_map = {item["image"]: item["captions"] for item in ground_truth_data}
image_names = list(gt_map.keys())

# Batch processing
batch_size = 250  # You’ll get 4 JSON files (1000 / 250)
total_batches = (len(image_names) + batch_size - 1) // batch_size


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/503 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/707M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/mini_project/subset/subset_1k_data.json'

In [ ]:

for batch_idx in range(total_batches):
    start = batch_idx * batch_size
    end = min((batch_idx + 1) * batch_size, len(image_names))
    batch_image_names = image_names[start:end]

    print(f"\n🔄 Processing batch {batch_idx + 1}/{total_batches} ({start} to {end - 1})")

    results = []
    for img_name in tqdm(batch_image_names):
        img_path = os.path.join(image_dir, img_name)
        try:
            image = Image.open(img_path).convert("RGB")
            tensor = transform(image).unsqueeze(0).to(device)

            result = {
                "image": img_name,
                "ground_truth_captions": gt_map[img_name],
                "original_caption": generate_caption(tensor),
                "fgsm": generate_caption(fgsm_attack(tensor.clone())),
                "pgd": generate_caption(pgd_attack(tensor.clone())),
                "cw": generate_caption(cw_attack(tensor.clone())),
                "jsma": generate_caption(jsma_attack(tensor.clone())),
                "one_pixel": generate_caption(one_pixel_attack(tensor.clone())),
            }

            results.append(result)

        except Exception as e:
            print(f"❌ Error with {img_name}: {e}")

    # Save batch output
    batch_output_path = f"{output_base_path}_{batch_idx + 1}.json"
    with open(batch_output_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"✅ Saved batch {batch_idx + 1} results to: {batch_output_path}")



🔄 Processing batch 1/4 (0 to 249)


100%|██████████| 250/250 [05:58<00:00,  1.43s/it]


✅ Saved batch 1 results to: /content/drive/MyDrive/adversarial_caption_output_batch_1.json

🔄 Processing batch 2/4 (250 to 499)


100%|██████████| 250/250 [05:33<00:00,  1.33s/it]


✅ Saved batch 2 results to: /content/drive/MyDrive/adversarial_caption_output_batch_2.json

🔄 Processing batch 3/4 (500 to 749)


100%|██████████| 250/250 [05:32<00:00,  1.33s/it]


✅ Saved batch 3 results to: /content/drive/MyDrive/adversarial_caption_output_batch_3.json

🔄 Processing batch 4/4 (750 to 999)


100%|██████████| 250/250 [05:41<00:00,  1.36s/it]

✅ Saved batch 4 results to: /content/drive/MyDrive/adversarial_caption_output_batch_4.json


In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:

!git clone https://github.com/salaniz/pycocoevalcap


Cloning into 'pycocoevalcap'...
remote: Enumerating objects: 821, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 821 (delta 4), reused 3 (delta 3), pack-reused 809 (from 2)
Receiving objects: 100% (821/821), 130.06 MiB | 37.56 MiB/s, done.
Resolving deltas: 100% (424/424), done.


In [ ]:

!pip install pycocoevalcap/


Processing ./pycocoevalcap
  Preparing metadata (setup.py) ... done
  Created wheel for pycocoevalcap: filename=pycocoevalcap-1.2-py3-none-any.whl size=104312245 sha256=15b4dc699a75d388089775350ad2c014164cf9fb4eda51c8990a004d9e025cb0
  Stored in directory: /tmp/pip-ephem-wheel-cache-vz57j8u9/wheels/0e/98/9f/b6578f2310a0adf702387edf950a2ba69dbf680c0b6830b312
Successfully built pycocoevalcap


In [ ]:

import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.cider.cider import Cider
from bert_score import score as bert_score


# Load your JSON file
with open("merged_output.json") as f:
    data = json.load(f)

smoother = SmoothingFunction().method4
meteor_scorer = Meteor()
cider_scorer = Cider()

# Initialize storage for scores
scores = {
    "original": {"BLEU": [], "METEOR": [], "CIDEr": []},
    "fgsm": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "pgd": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "cw": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "jsma": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "one_pixel": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []}
}

# List of attacks to evaluate
attacks = ["fgsm", "pgd", "cw", "jsma", "one_pixel"]


In [ ]:

# Evaluate each item in the dataset
print("🔍 Starting evaluation...")
for idx, item in enumerate(data, 1):  # Start count from 1
    refs = [caption.lower().split() for caption in item["ground_truth_captions"]]

    # Original Caption
    orig = item["original_caption"].lower().split()
    scores["original"]["BLEU"].append(sentence_bleu(refs, orig, smoothing_function=smoother))
    scores["original"]["METEOR"].append(
        meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )
    scores["original"]["CIDEr"].append(
        cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )

    # Attacks
    for atk in attacks:
        adv_cap = item[f"{atk}"]
        adv_tok = adv_cap.lower().split()

        scores[atk]["BLEU"].append(sentence_bleu(refs, adv_tok, smoothing_function=smoother))
        scores[atk]["METEOR"].append(
            meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )
        scores[atk]["CIDEr"].append(
            cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )

        # BERTScore: compare with original
        P, R, F1 = bert_score([adv_cap], [item["original_caption"]], lang="en", verbose=False)
        scores[atk]["BERT"].append(F1.item())

    # Progress print every 50 items (customizable)
    if idx % 50 == 0 or idx == len(data):
        print(f"✅ Processed {idx}/{len(data)} images")



🔍 Starting evaluation...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 50/1000 images


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 100/1000 images


...

✅ Processed 900/1000 images


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 950/1000 images


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 1000/1000 images


In [ ]:

# Reporting averages
def avg(lst): return round(np.mean(lst), 4)


print("📊 Robustness Evaluation Summary:\n")

print("Original:")
print(f"  BLEU: {avg(scores['original']['BLEU'])}")
print(f"  METEOR: {avg(scores['original']['METEOR'])}")
print(f"  CIDEr: {avg(scores['original']['CIDEr'])}")

for atk in attacks:
    print(f"\n⚠ Under {atk.upper()} Attack:")
    print(f"  BLEU: {avg(scores[atk]['BLEU'])} (drop: {round(avg(scores['original']['BLEU']) - avg(scores[atk]['BLEU']), 4)})")
    print(f"  METEOR: {avg(scores[atk]['METEOR'])} (drop: {round(avg(scores['original']['METEOR']) - avg(scores[atk]['METEOR']), 4)})")
    print(f"  CIDEr: {avg(scores[atk]['CIDEr'])} (drop: {round(avg(scores['original']['CIDEr']) - avg(scores[atk]['CIDEr']), 4)})")
    print(f"  BERTScore F1 vs Original: {avg(scores[atk]['BERT'])}")

📊 Robustness Evaluation Summary:

Original:
  BLEU: 0.1528
  METEOR: 0.1815
  CIDEr: 0.0

⚠ Under FGSM Attack:
  BLEU: 0.1535 (drop: -0.0007)
  METEOR: 0.1824 (drop: -0.0009)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.9926

⚠ Under PGD Attack:
  BLEU: 0.1487 (drop: 0.0041)
  METEOR: 0.1795 (drop: 0.002)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.9749

⚠ Under CW Attack:
  BLEU: 0.1266 (drop: 0.0262)
  METEOR: 0.1702 (drop: 0.0113)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.9184

⚠ Under JSMA Attack:
  BLEU: 0.1529 (drop: -0.0001)
  METEOR: 0.1819 (drop: -0.0004)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.9862

⚠ Under ONE_PIXEL Attack:
  BLEU: 0.1537 (drop: -0.0009)
  METEOR: 0.1815 (drop: 0.0)
  CIDEr: 0.0 (drop: 0.0)
  BERTScore F1 vs Original: 0.996
